# Avendi V8-Y2 — Kaggle 2×T4 Source-Pinned Campaign Notebook

- **Pinned implementation commit (`SOURCE_COMMIT`)**: `428664245c11400464a050fce4422ade4b3a1b76`
- **Design registry digest (`DESIGN_SHA`)**: `13f8f46c9065115b9bbdecb2884edba2ff88a3bf9e01cba2afd613739c344fbd`
- **Default phase**: `PHASE = "frozen"` (120 wall-minute ceiling; executes P0 CPU instrument qualification + P1 Y1 checkpoint restoration & parity join + P2 frozen crossover diagnosis + stage routing receipt).
- **Missing-checkpoint resilience**: If `ELENDRIL_V8_Y1_FULL.zip` is not attached, the notebook still completes P0 CPU qualification and emits a valid `frozen_receipt.json` with an explicit missing-checkpoint manifest and deterministic retraining recipe.
- **Top-level `/kaggle/working/` exports**: All exported `.zip` bundles and `FINAL_STATUS.json` are written to `/kaggle/working/avendi_v8_y2_<PHASE>/exports/` AND mirrored directly to `/kaggle/working/` top level for one-click download.

In [ ]:
from hashlib import sha256
import json, os, shutil, subprocess, sys
from pathlib import Path

OWNER_ENABLED = True
PHASE = "frozen"  # 'fixture', 'frozen' (default), 'acquisition', 'transition', 'conditional', 'correction', 'all'
MINUTES_BUDGET = 120  # frozen=120, acquisition=240, transition=240
ADMISSION_RECEIPT = None  # Required when PHASE in ('acquisition', 'transition', 'conditional', 'correction')

SOURCE_COMMIT = "428664245c11400464a050fce4422ade4b3a1b76"
DESIGN_SHA = "13f8f46c9065115b9bbdecb2884edba2ff88a3bf9e01cba2afd613739c344fbd"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
EXPECTED_Y1_EVIDENCE_SHA = "4f76ac0d3a1fc67fac2705325c0909f791325696f9a1162c9e86c783c717173f"
EXPECTED_Y1_SUMMARY_SHA = "9844ad2015c6c3c9b9bf8ffd8864b8bded9d37ec001c89bc73ac4f64c8cad1bf"
EXPECTED_HH_FULL_SHA = "43a0c074cdcc3e12d8d00e0509663a2d47702586414317818ca0876579e487e2"
EXPECTED_Y1_LINEAGES = {"Y1-t1_factorial-S3-F00-s7": "57ab8b7292a84689ee32d797257eb4a2c9c8cd3c95421df2ca254801c6cd1706", "Y1-t1_factorial-S3-F00-s19": "b900fa477499e5a7eca2d5ac4ed717e1f1ff34981a3ae2a85e9aae1a8c3da475", "Y1-t1_factorial-S3-F00-s41": "4e8891f6f3d8ef1975d949bc7c96393edda9c4dc304b12cf5a5368e5fdffdf95", "Y1-t1_factorial-S3-F11-s7": "65c8506d7359d91c76fd5ac3628f0226c68f76a2edf760f1e62d309e9fa0313f", "Y1-t1_factorial-S3-F11-s19": "96fa4f25b62601b8533d602ad5e1ba6587760bc3cb8bb758a0230a2a7bf686ef", "Y1-t1_factorial-S3-F11-s41": "f2ce4dfc965559ee3b2e189a80ce3e8e84786c4e23d1de3a78249b544961babf"}

def _sha256(path: Path) -> str:
    h = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def discover_mounted_inputs() -> dict:
    root = Path("/kaggle/input")
    found = {"y1_checkpoint_root": None, "y1_evidence": None, "hh_full": None}
    if not root.exists():
        return found
    for p in sorted(root.rglob("*.zip")):
        if not p.is_file():
            continue
        if p.name == "ELENDRIL_V8_Y1_FULL.zip":
            found["y1_checkpoint_root"] = str(p)
        elif p.name == "ELENDRIL_V8_Y1_EVIDENCE.zip" and _sha256(p) == EXPECTED_Y1_EVIDENCE_SHA:
            found["y1_evidence"] = str(p)
        elif p.name == "ELENDRIL_HITCHHIKER_FULL.zip" and _sha256(p) == EXPECTED_HH_FULL_SHA:
            found["hh_full"] = str(p)
    if found["y1_checkpoint_root"] is None:
        for d in sorted(root.rglob("checkpoints")):
            if d.is_dir() and any((d / jid).is_dir() for jid in EXPECTED_Y1_LINEAGES):
                found["y1_checkpoint_root"] = str(d)
                break
    if found["y1_evidence"] is None:
        for ipath in sorted(root.rglob("identity.json")):
            if ipath.parent.name == "analysis" and (ipath.parent.parent / "prepared" / "development_suite.json").is_file():
                found["y1_evidence"] = str(ipath.parent.parent)
                break
    return found

DISCOVERED = discover_mounted_inputs()
WORK_ROOT = Path(f"/kaggle/working/avendi_v8_y2_{PHASE}")
CODE_ROOT = Path("/kaggle/working/anra_avendi_src")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CODE_ROOT)], check=True)
subprocess.run(["git", "-C", str(CODE_ROOT), "fetch", "--all", "--tags"], check=True)
resolved = subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", SOURCE_COMMIT], text=True).strip()
assert resolved == SOURCE_COMMIT, f"commit mismatch: {resolved} != {SOURCE_COMMIT}"
subprocess.run(["git", "-C", str(CODE_ROOT), "checkout", "--detach", SOURCE_COMMIT], check=True)
os.chdir(CODE_ROOT)
print(json.dumps({"source": SOURCE_COMMIT, "design_sha256": DESIGN_SHA, "phase": PHASE, "discovered": DISCOVERED}, indent=2))


In [ ]:
# Step 1: Run P0 CPU instrument qualification first
p0_cmd = [
    sys.executable, "-m", "tools.avendi_campaign",
    "--phase", "fixture",
    "--source", SOURCE_COMMIT,
    "--output", str(WORK_ROOT / "p0_preflight"),
]
subprocess.run(p0_cmd, cwd=CODE_ROOT, check=True)

# Step 2: Execute requested campaign phase (default: 'frozen')
cmd = [
    sys.executable, "-m", "tools.avendi_campaign",
    "--phase", PHASE,
    "--source", SOURCE_COMMIT,
    "--output", str(WORK_ROOT),
    "--minutes", str(MINUTES_BUDGET),
]
if OWNER_ENABLED:
    cmd.append("--owner-enabled")
if DISCOVERED.get("y1_checkpoint_root"):
    cmd.extend(["--checkpoint-root", str(DISCOVERED["y1_checkpoint_root"])])
if DISCOVERED.get("y1_evidence"):
    cmd.extend(["--evidence", str(DISCOVERED["y1_evidence"])])
if ADMISSION_RECEIPT is not None:
    cmd.extend(["--admission", str(ADMISSION_RECEIPT)])

print("Launching:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=CODE_ROOT)

# Step 3: Run offline analyzer to emit RESULTS.md & METRICS.json
ev_zip = WORK_ROOT / "exports" / "AVENDI_V8_Y2_EVIDENCE.zip"
if ev_zip.is_file():
    subprocess.run(
        [sys.executable, "-m", "tools.analyze_avendi", "--evidence", str(ev_zip), "--output", str(WORK_ROOT / "report")],
        cwd=CODE_ROOT,
        check=False,
    )


In [ ]:
# Mirror exports directly into /kaggle/working/ root so they appear at the very top of Kaggle's Output sidebar
top_root = Path("/kaggle/working")
for p in sorted((WORK_ROOT / "exports").glob("*")):
    if p.is_file():
        dest = top_root / p.name
        if dest.resolve() != p.resolve():
            shutil.copy2(p, dest)
        print(f"EXPORT: {dest} ({dest.stat().st_size / (1024 * 1024):.2f} MB)")
        try:
            from IPython.display import FileLink, display
            display(FileLink(p.name))
        except ImportError:
            pass

for rfile in (WORK_ROOT / "report" / "RESULTS.md", WORK_ROOT / "report" / "METRICS.json"):
    if rfile.is_file():
        dest = top_root / f"AVENDI_V8_Y2_{rfile.name}"
        shutil.copy2(rfile, dest)
        print(f"REPORT: {dest}")
